# Data Wrangling with LLM data (logits extracted for 1500 human answers (in context), then weighted with human answers)
Starting point:
- 848 datasets (18 tasks per model (minus the NON_IDEAL_OUTPUTS)) with all logprobs for all answer alternatives of each subtask for all ~1.500 participants. 

What does this script do
- Read all survey data sets
- normalise log-prob scores so that they are probabilities that sum up to 1 over all answer options
- filter out the probability that the LLM assigned to the answer option which the participant actually chose (for each item)
- weigh probabiliies with human answers and divide by all probabilities for that item, to have one number (in the realm, if not exact the number of the answer alternatives) per item per model
- second weighing strategy, where score of model per item is only weighed with top n most likely probabilities that model assigned
- for some scales: add subcategories (each item belongs to a subcategory)

- make several version:
    - one version where reverse reversed items, one without
    - flipped answers separately, flip back plus do not flip back

- concat the itemwise scores for each task per model all together in final data frames (divided by whether flipped or not, and whether reversed or not) and save them.

Specials:
- for every scale we compared the Frey et al. (2017) materials quest_raw.csv and quest_proc.csv and tried to trace back how they got from one to the other. Then we did the same transformations
- therefore, for some tasks the scores are mapped onto a different scale (point system depending on given answer) on the `human_number` level (before mapped to LLM assigned probabilities)

Goal:
- have one value per item per model


## Packages & Helpers

In [ ]:
# packages
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
from utils import load_dataframes, filter_pred_prob

PATH = "../../data/raw/risk_data/LLM_data/prompts_weighed/"

In [ ]:
# ----------------------------------------------------------------------------------------------
# Weighted score per model × item
# ----------------------------------------------------------------------------------------------

# Pro Modell × Item die Zähler und Nenner berechnen
# ----------------------------------------------------------------------------------------------
# - Zähler = Summe von (Antwort * Wahrscheinlichkeit)
# - Nenner = Summe von (Wahrscheinlichkeiten)

def compute_weighted_score(group):
    numerator = (group["human_number"] * group["prob_pred"]).sum()
    denominator = group["prob_pred"].sum()
    return numerator / denominator if denominator > 0 else None


# ----------------------------------------------------------------------------------------------
# Top-n weighted score
# ----------------------------------------------------------------------------------------------s

def compute_top_n_weighted_score(group, n=100):
    # Sort rows by probability descending
    top_n = group.sort_values("prob_pred", ascending=False).head(n)

    numerator = (top_n["human_number"] * top_n["prob_pred"]).sum()
    denominator = top_n["prob_pred"].sum()

    return numerator / denominator if denominator > 0 else None


# ----------------------------------------------------------------------------------------------
# One value per model × item 
# ----------------------------------------------------------------------------------------------

def _attach_reverse_coded(result, data):
    if "reverse_coded" in data.columns:
        reverse_map = data[["experiment", "item", "reverse_coded"]].drop_duplicates()
        result = result.merge(reverse_map, on=["experiment", "item"], how="left")
    return result


def get_LLM_value_per_item(data):
    grouped = data.groupby(["experiment", "model", "item"])

    score = grouped.apply(
        lambda g: (g["human_number"] * g["prob_pred"]).sum() / g["prob_pred"].sum()
    )

    result = score.reset_index(name="score")
    return _attach_reverse_coded(result, data)


# ----------------------------------------------------------------------------------------------
# One value per model × item 
# ----------------------------------------------------------------------------------------------

def get_LLM_value_per_item_top_n(data, n=100):
    result = (
        data.groupby(["experiment", "model", "item"])[["human_number", "prob_pred"]]
            .apply(lambda g: compute_top_n_weighted_score(g, n=n))
            .reset_index(name="score_top_n")
    )
    return result


## AUDIT SCALE

In [ ]:
# load data
AUDIT_data = load_dataframes(task_name="AUDIT", path = PATH)

In [ ]:
# normalise answer option sum to one (tun so als hätten wir sehr guten Prompt, dann würde LLM nur zwischen möglichen Antwortalternativen aussuchen, das simulieren wir dadurch)
mask = (AUDIT_data["item"] == 1)
AUDIT_data.loc[mask, "prob_1"] = np.exp(AUDIT_data.loc[mask, "1"])/(np.exp(AUDIT_data.loc[mask, "1"]) + np.exp(AUDIT_data.loc[mask, "2"]))
AUDIT_data.loc[mask, "prob_2"] = np.exp(AUDIT_data.loc[mask, "2"])/(np.exp(AUDIT_data.loc[mask, "1"]) + np.exp(AUDIT_data.loc[mask, "2"]))

mask = (AUDIT_data["item"].isin([10, 11]))
AUDIT_data.loc[mask, "prob_1"] = np.exp(AUDIT_data.loc[mask, "1"])/(np.exp(AUDIT_data.loc[mask, "1"]) + np.exp(AUDIT_data.loc[mask, "2"]) + np.exp(AUDIT_data.loc[mask, "3"]))
AUDIT_data.loc[mask, "prob_2"] = np.exp(AUDIT_data.loc[mask, "2"])/(np.exp(AUDIT_data.loc[mask, "1"]) + np.exp(AUDIT_data.loc[mask, "2"]) + np.exp(AUDIT_data.loc[mask, "3"]))
AUDIT_data.loc[mask, "prob_3"] = np.exp(AUDIT_data.loc[mask, "3"])/(np.exp(AUDIT_data.loc[mask, "1"]) + np.exp(AUDIT_data.loc[mask, "2"]) + np.exp(AUDIT_data.loc[mask, "3"]))


mask = (AUDIT_data["item"].isin([2, 3, 4, 5, 6, 7, 8, 9]))
AUDIT_data.loc[mask, "prob_1"] = np.exp(AUDIT_data.loc[mask, "1"])/(np.exp(AUDIT_data.loc[mask, "1"]) + np.exp(AUDIT_data.loc[mask, "2"]) + np.exp(AUDIT_data.loc[mask, "3"]) + np.exp(AUDIT_data.loc[mask, "4"]) + np.exp(AUDIT_data.loc[mask, "5"]))
AUDIT_data.loc[mask, "prob_2"] = np.exp(AUDIT_data.loc[mask, "2"])/(np.exp(AUDIT_data.loc[mask, "1"]) + np.exp(AUDIT_data.loc[mask, "2"]) + np.exp(AUDIT_data.loc[mask, "3"]) + np.exp(AUDIT_data.loc[mask, "4"]) + np.exp(AUDIT_data.loc[mask, "5"]))
AUDIT_data.loc[mask, "prob_3"] = np.exp(AUDIT_data.loc[mask, "3"])/(np.exp(AUDIT_data.loc[mask, "1"]) + np.exp(AUDIT_data.loc[mask, "2"]) + np.exp(AUDIT_data.loc[mask, "3"]) + np.exp(AUDIT_data.loc[mask, "4"]) + np.exp(AUDIT_data.loc[mask, "5"]))
AUDIT_data.loc[mask, "prob_4"] = np.exp(AUDIT_data.loc[mask, "4"])/(np.exp(AUDIT_data.loc[mask, "1"]) + np.exp(AUDIT_data.loc[mask, "2"]) + np.exp(AUDIT_data.loc[mask, "3"]) + np.exp(AUDIT_data.loc[mask, "4"]) + np.exp(AUDIT_data.loc[mask, "5"]))
AUDIT_data.loc[mask, "prob_5"] = np.exp(AUDIT_data.loc[mask, "5"])/(np.exp(AUDIT_data.loc[mask, "1"]) + np.exp(AUDIT_data.loc[mask, "2"]) + np.exp(AUDIT_data.loc[mask, "3"]) + np.exp(AUDIT_data.loc[mask, "4"]) + np.exp(AUDIT_data.loc[mask, "5"]))


In [ ]:
# filter out probability LLM assigned to real item answer 
AUDIT_data=filter_pred_prob(AUDIT_data)

In [ ]:
# add whether item was reverse coded
reverse_coded = {1: True, 2: False, 3: False,  4: False, 5: False,  6: False,  7: False,  8: False,  9: False,  10: False, 11: False}
# Apply mapping row-wise based on item number
AUDIT_data["reverse_coded"] = AUDIT_data["item"].map(reverse_coded)

In [ ]:
# seperate the datasets
raw_audit = AUDIT_data[AUDIT_data["flipped"] == "no"]
raw_audit_flipped = AUDIT_data[AUDIT_data["flipped"] == "yes"]
audit_flipped_back = raw_audit_flipped.copy()


In [ ]:
# flip back human answers where they were flipped
mask = (audit_flipped_back["item"] == 1)
audit_flipped_back.loc[mask, "human_number"] = 3 - audit_flipped_back.loc[mask, "human_number"]
mask = (audit_flipped_back["item"].isin([10, 11]))
audit_flipped_back.loc[mask, "human_number"] = 4 - audit_flipped_back.loc[mask, "human_number"]
mask = (audit_flipped_back["item"].isin([2, 3, 4, 5, 6, 7, 8, 9]))
audit_flipped_back.loc[mask, "human_number"] = 6 - audit_flipped_back.loc[mask, "human_number"]


In [ ]:
# Define mappings for each AUDIT question:
audit_maps = {
    1: {1: 1, 2: 0},                            # AlcSplit
    2: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4},          # AUDIT_1 (in proc data Frey)
    3: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4},          # AUDIT_2 (in proc data Frey)
    4: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4},          # AUDIT_3 (in proc data Frey)
    5: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4},          # AUDIT_4 (in proc data Frey)
    6: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4},          # AUDIT_5 (in proc data Frey)
    7: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4},          # AUDIT_6 (in proc data Frey)
    8: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4},          # AUDIT_7 (in proc data Frey)
    9: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4},          # AUDIT_8 (in proc data Frey)
    10: {1: 0, 2: 2, 3: 4},                     # AUDIT_10 (in proc data Frey)
    11: {1: 0, 2: 2, 3: 4},                     # AUDIT_11 (in proc data Frey)

}

# Apply mapping row-wise based on item number
def recode_audit(row):
    mapping = audit_maps.get(row["item"])
    if mapping is not None:
        return mapping.get(row["human_number"], None)  # None if invalid code
    return row["human_number"]  


audit_reversed = raw_audit.copy()
audit_reversed["human_number"] = audit_reversed.apply(recode_audit, axis=1)

audit_flipped_back_reversed_back = audit_flipped_back.copy()
audit_flipped_back_reversed_back["human_number"] = audit_flipped_back_reversed_back.apply(recode_audit, axis=1)


In [ ]:
# produce df with one value per model per item 
model_item_scores_AUDIT_raw = get_LLM_value_per_item(raw_audit)
model_item_scores_AUDIT_top_n_raw = get_LLM_value_per_item_top_n(raw_audit)

# Merge them on the grouping keys
model_item_scores_AUDIT_raw = model_item_scores_AUDIT_raw.merge(
    model_item_scores_AUDIT_top_n_raw,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_AUDIT_reversed = get_LLM_value_per_item(audit_reversed)
model_item_scores_AUDIT_top_n_reversed = get_LLM_value_per_item_top_n(audit_reversed)

# Merge them on the grouping keys
model_item_scores_AUDIT_reversed = model_item_scores_AUDIT_reversed.merge(
    model_item_scores_AUDIT_top_n_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_AUDIT_raw_flipped = get_LLM_value_per_item(raw_audit_flipped)
model_item_scores_AUDIT_top_n_raw_flipped = get_LLM_value_per_item_top_n(raw_audit_flipped)

# Merge them on the grouping keys
model_item_scores_AUDIT_raw_flipped = model_item_scores_AUDIT_raw_flipped.merge(
    model_item_scores_AUDIT_top_n_raw_flipped,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_AUDIT_flipped_reversed = get_LLM_value_per_item(audit_flipped_back_reversed_back)
model_item_scores_AUDIT_top_n_flipped_reversed = get_LLM_value_per_item_top_n(audit_flipped_back_reversed_back)

# Merge them on the grouping keys
model_item_scores_AUDIT_flipped_reversed = model_item_scores_AUDIT_flipped_reversed.merge(
    model_item_scores_AUDIT_top_n_flipped_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = model_item_scores_AUDIT_reversed
no_flip_data_raw = model_item_scores_AUDIT_raw

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = model_item_scores_AUDIT_flipped_reversed
flip_data_raw = model_item_scores_AUDIT_raw_flipped

## BARRATT SCALE

In [ ]:
# load data
BARRATT_data = load_dataframes(task_name="BARRAT", path = PATH)

In [ ]:
# normalise answer option sum to one
BARRATT_data["prob_1"] = np.exp(BARRATT_data["1"])/(np.exp(BARRATT_data["1"]) + np.exp(BARRATT_data["2"]) + np.exp(BARRATT_data["3"]) + np.exp(BARRATT_data["4"]))
BARRATT_data["prob_2"] = np.exp(BARRATT_data["2"])/(np.exp(BARRATT_data["1"]) + np.exp(BARRATT_data["2"]) + np.exp(BARRATT_data["3"]) + np.exp(BARRATT_data["4"]))
BARRATT_data["prob_3"] = np.exp(BARRATT_data["3"])/(np.exp(BARRATT_data["1"]) + np.exp(BARRATT_data["2"]) + np.exp(BARRATT_data["3"]) + np.exp(BARRATT_data["4"]))
BARRATT_data["prob_4"] = np.exp(BARRATT_data["4"])/(np.exp(BARRATT_data["1"]) + np.exp(BARRATT_data["2"]) + np.exp(BARRATT_data["3"]) + np.exp(BARRATT_data["4"]))


In [ ]:
# filter out probability LLM assigned to real item answer 
BARRATT_data=filter_pred_prob(BARRATT_data)

In [ ]:
# add whether item was reverse coded
reverse_coded = {
    1: True, 2: False, 3: False,  4: False, 5: False,  6: False,  7: True,  8: True,  9: True,  10: True,
    11: False, 12: True, 13: True,  14: False, 15: True,  16: False,  17: False,  18: False,  19: False,  20: True,
    21: False, 22: False, 23: False,  24: False, 25: False,  26: False,  27: False,  28: False,  29: True,  30: True
    }
# Apply mapping row-wise based on item number
BARRATT_data["reverse_coded"] = BARRATT_data["item"].map(reverse_coded)

In [ ]:
# seperate the datasets
raw_barrat = BARRATT_data[BARRATT_data["flipped"] == False]
raw_barrat_flipped = BARRATT_data[BARRATT_data["flipped"] == True]
barrat_flipped_back = raw_barrat_flipped.copy()
barrat_flipped_back["human_number"] = 5 - barrat_flipped_back["human_number"]

In [ ]:
# reverse LLM answers (again) where the items where reversed phrased
barrat_reversed = raw_barrat.copy()

# flip back answers that where reverse coded
mask = (barrat_reversed["reverse_coded"] == True)
barrat_reversed.loc[mask, "human_number"] = 5 - barrat_reversed.loc[mask, "human_number"]


# ------ for re-flipped data ------------
# Apply mapping row-wise based on item number
barrat_flipped_back_reversed_back = barrat_flipped_back.copy()

# flip back answers that where reverse coded
mask = (barrat_flipped_back_reversed_back["reverse_coded"] == True)
barrat_flipped_back_reversed_back.loc[mask, "human_number"] = 5 - barrat_flipped_back_reversed_back.loc[mask, "human_number"]


In [ ]:
# produce df with one value per model per item 
model_item_scores_BARRATT_raw = get_LLM_value_per_item(raw_barrat)
model_item_scores_BARRATT_top_n_raw = get_LLM_value_per_item_top_n(raw_barrat)

# Merge them on the grouping keys
model_item_scores_BARRATT_raw = model_item_scores_BARRATT_raw.merge(
    model_item_scores_BARRATT_top_n_raw,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_BARRATT_reversed = get_LLM_value_per_item(barrat_reversed)
model_item_scores_BARRATT_top_n_reversed = get_LLM_value_per_item_top_n(barrat_reversed)

# Merge them on the grouping keys
model_item_scores_BARRATT_reversed = model_item_scores_BARRATT_reversed.merge(
    model_item_scores_BARRATT_top_n_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_BARRATT_raw_flipped = get_LLM_value_per_item(raw_barrat_flipped)
model_item_scores_BARRATT_top_n_raw_flipped = get_LLM_value_per_item_top_n(raw_barrat_flipped)

# Merge them on the grouping keys
model_item_scores_BARRATT_raw_flipped = model_item_scores_BARRATT_raw_flipped.merge(
    model_item_scores_BARRATT_top_n_raw_flipped,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_BARRATT_flipped_reversed = get_LLM_value_per_item(barrat_flipped_back_reversed_back)
model_item_scores_BARRATT_top_n_flipped_reversed = get_LLM_value_per_item_top_n(barrat_flipped_back_reversed_back)

# Merge them on the grouping keys
model_item_scores_BARRATT_flipped_reversed = model_item_scores_BARRATT_flipped_reversed.merge(
    model_item_scores_BARRATT_top_n_flipped_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

In [ ]:
# Adding task specific categories to save in all data

# add item categories
item_to_category = {
    1: "BISn", 2: "BISm", 3: "BISm",  4: "BISm", 5: "BISa",  6: "BISa",  7: "BISn",  8: "BISn",  9: "BISa",  10: "BISn",
    11: "BISa", 12: "BISn", 13: "BISn",  14: "BISn", 15: "BISn",  16: "BISm",  17: "BISm",  18: "BISn",  19: "BISm",  20: "BISa",
    21: "BISm", 22: "BISm", 23: "BISm",  24: "BISa", 25: "BISm",  26: "BISa",  27: "BISn",  28: "BISa",  29: "BISn",  30: "BISm"
}

model_item_scores_BARRATT_raw["category"] = model_item_scores_BARRATT_raw["item"].map(item_to_category)
model_item_scores_BARRATT_reversed["category"] = model_item_scores_BARRATT_reversed["item"].map(item_to_category)
model_item_scores_BARRATT_raw_flipped["category"] = model_item_scores_BARRATT_raw_flipped["item"].map(item_to_category)
model_item_scores_BARRATT_flipped_reversed["category"] = model_item_scores_BARRATT_flipped_reversed["item"].map(item_to_category)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_BARRATT_reversed], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_BARRATT_raw], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, model_item_scores_BARRATT_flipped_reversed], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, model_item_scores_BARRATT_raw_flipped], ignore_index=True)

## CARE TASK

In [ ]:
# load data
CARE_data = load_dataframes(task_name="CARE", path = PATH)

In [ ]:
# get probabilities out of log-probabilities

cols = [str(i) for i in range(0, 100)]
# Compute normalized probabilities
exp_vals = np.exp(CARE_data[cols])
prob_vals = exp_vals.div(exp_vals.sum(axis=1), axis=0)

# Rename columns all at once
prob_vals.columns = [f"prob_{i}" for i in range(0, 100)]

# Join to original dataframe in one step
CARE_data = pd.concat([CARE_data, prob_vals], axis=1).copy()

In [ ]:
# filter out probability LLM assigned to real item answer 
CARE_data=filter_pred_prob(CARE_data)

In [ ]:
# add whether item was reverse coded, none were!
# False for all
CARE_data["reverse_coded"] = False

In [ ]:
# produce df with one value per model per item 
model_item_scores_CARE = get_LLM_value_per_item(CARE_data)
model_item_scores_CARE_top_n = get_LLM_value_per_item_top_n(CARE_data)

# Merge them on the grouping keys
model_item_scores_CARE = model_item_scores_CARE.merge(
    model_item_scores_CARE_top_n,
    on=["experiment", "model", "item"],
    how="inner" 
)

In [ ]:
# Adding task specific categories to save in all data
# add item categories
item_to_category = {
    1: "CAREa", 2: "CAREa", 3: "CAREa",  4: "CAREa", 5: "CAREa",  6: "CAREa",  7: "CAREa",  8: "CAREa",  9: "CAREa",  10: "CAREs",
    11: "CAREs", 12: "CAREs", 13: "CAREs",  14: "CAREs", 15: "CAREs",  16: "CAREw",  17: "CAREw",  18: "CAREw",  19: "CAREw"
}

model_item_scores_CARE["category"] = model_item_scores_CARE["item"].map(item_to_category)


In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_CARE], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_CARE], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
# no flipped data

## DAST SCALE

In [ ]:
# load data
DAST_data = load_dataframes(task_name="DAST", path = PATH)

In [ ]:
# normalise answer option sum to one 
DAST_data["prob_1"] = np.exp(DAST_data["1"])/(np.exp(DAST_data["1"]) + np.exp(DAST_data["2"]))
DAST_data["prob_2"] = np.exp(DAST_data["2"])/(np.exp(DAST_data["1"]) + np.exp(DAST_data["2"]))

In [ ]:
# filter out probability LLM assigned to real item answer 
DAST_data=filter_pred_prob(DAST_data)

In [ ]:
# add whether item was reverse coded
reverse_coded = {
    1: True, 2: True, 3: True,  4: False, 5: False,  6: True,  7: True,  8: True,  9: True,  10: True, 
    11: True, 12: True, 13: True,  14: True, 15: True,  16: True,  17: True,  18: True,  19: True,  20: True
    }

# Apply mapping row-wise based on item number
DAST_data["reverse_coded"] = DAST_data["item"].map(reverse_coded)



In [ ]:
# seperate the datasets
raw_dast = DAST_data[DAST_data["flipped"] == False]
raw_dast_flipped = DAST_data[DAST_data["flipped"] == True]
dast_flipped_back = raw_dast_flipped.copy()
# flip back human answers where they were flipped
dast_flipped_back["human_number"] = 3 - dast_flipped_back["human_number"]

In [ ]:
# Define mappings for each DAST question:
dast_maps = {
    1: {1: 1, 2: 0},                           
    2: {1: 1, 2: 0},
    3: {1: 1, 2: 0},
    4: {1: 0, 2: 1},
    5: {1: 0, 2: 1},
    6: {1: 1, 2: 0},
    7: {1: 1, 2: 0},
    8: {1: 1, 2: 0},
    9: {1: 1, 2: 0},
    10: {1: 1, 2: 0},
    11: {1: 1, 2: 0},
    12: {1: 1, 2: 0},
    13: {1: 1, 2: 0},
    14: {1: 1, 2: 0},
    15: {1: 1, 2: 0},
    16: {1: 1, 2: 0},
    17: {1: 1, 2: 0},
    18: {1: 1, 2: 0},
    19: {1: 1, 2: 0},
    20: {1: 1, 2: 0}
}

# Apply mapping row-wise based on item number
def recode_dast(row):
    mapping = dast_maps.get(row["item"])
    if mapping is not None:
        return mapping.get(row["human_number"], None)  # None if invalid code
    return row["human_number"]  


dast_reversed = raw_dast.copy()
dast_reversed["human_number"] = dast_reversed.apply(recode_dast, axis=1)

dast_flipped_back_reversed_back = dast_flipped_back.copy()
dast_flipped_back_reversed_back["human_number"] = dast_flipped_back_reversed_back.apply(recode_dast, axis=1)


In [ ]:
# produce df with one value per model per item 
model_item_scores_DAST_raw = get_LLM_value_per_item(raw_dast)
model_item_scores_DAST_top_n_raw = get_LLM_value_per_item_top_n(raw_dast)

# Merge them on the grouping keys
model_item_scores_DAST_raw = model_item_scores_DAST_raw.merge(
    model_item_scores_DAST_top_n_raw,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_DAST_reversed = get_LLM_value_per_item(dast_reversed)
model_item_scores_DAST_top_n_reversed = get_LLM_value_per_item_top_n(dast_reversed)

# Merge them on the grouping keys
model_item_scores_DAST_reversed = model_item_scores_DAST_reversed.merge(
    model_item_scores_DAST_top_n_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_DAST_raw_flipped = get_LLM_value_per_item(raw_dast_flipped)
model_item_scores_DAST_top_n_raw_flipped = get_LLM_value_per_item_top_n(raw_dast_flipped)

# Merge them on the grouping keys
model_item_scores_DAST_raw_flipped = model_item_scores_DAST_raw_flipped.merge(
    model_item_scores_DAST_top_n_raw_flipped,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_DAST_flipped_reversed = get_LLM_value_per_item(dast_flipped_back_reversed_back)
model_item_scores_DAST_top_n_flipped_reversed = get_LLM_value_per_item_top_n(dast_flipped_back_reversed_back)

# Merge them on the grouping keys
model_item_scores_DAST_flipped_reversed = model_item_scores_DAST_flipped_reversed.merge(
    model_item_scores_DAST_top_n_flipped_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_DAST_reversed], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_DAST_raw], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, model_item_scores_DAST_flipped_reversed], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, model_item_scores_DAST_raw_flipped], ignore_index=True)

## DM SCALE

In [ ]:
# load data
DM_data = load_dataframes(task_name="DM", path = PATH)

In [ ]:
# normalise answer option sum to one
DM_data["prob_1"] = np.exp(DM_data["1"])/(np.exp(DM_data["1"]) + np.exp(DM_data["2"]) + np.exp(DM_data["3"]) + np.exp(DM_data["4"]))
DM_data["prob_2"] = np.exp(DM_data["2"])/(np.exp(DM_data["1"]) + np.exp(DM_data["2"]) + np.exp(DM_data["3"]) + np.exp(DM_data["4"]))
DM_data["prob_3"] = np.exp(DM_data["3"])/(np.exp(DM_data["1"]) + np.exp(DM_data["2"]) + np.exp(DM_data["3"]) + np.exp(DM_data["4"]))
DM_data["prob_4"] = np.exp(DM_data["4"])/(np.exp(DM_data["1"]) + np.exp(DM_data["2"]) + np.exp(DM_data["3"]) + np.exp(DM_data["4"]))


In [ ]:
# filter out probability LLM assigned to real item answer 
DM_data=filter_pred_prob(DM_data)

In [ ]:
# add whether item was reverse coded, none were!
# False for all
DM_data["reverse_coded"] = False # wobei inhaltlich bissl unklar, 4 wird jeweils zu 1, der Rest bleibt gleich.

In [ ]:
# seperate the datasets
raw_dm = DM_data[DM_data["flipped"] == False]
raw_dm_flipped = DM_data[DM_data["flipped"] == True]
dm_flipped_back = raw_dm_flipped.copy()
dm_flipped_back["human_number"] = 5 - dm_flipped_back["human_number"]

In [ ]:
# Define mappings for DM, so that all 4s are transformed to 1s (like in original Fey dataset):
# hier Abweichung von sonst Orientierung an Frey quest_proc df, aber da sonst später Umwandlung, hier gleich zu Scale 0-2
mapping = {
    4: 1,
    3: 2,
    2: 1,
    1: 0
} # here, I already go one step further than human process, directly to real scores that are then summed/averaged for final sum

dm_reversed = raw_dm.copy()
dm_reversed["human_number"] = dm_reversed["human_number"].map(mapping)

dm_flipped_back_reversed_back = dm_flipped_back.copy()
dm_flipped_back_reversed_back["human_number"] = dm_flipped_back_reversed_back["human_number"].map(mapping)


In [ ]:
# produce df with one value per model per item 
model_item_scores_DM_raw = get_LLM_value_per_item(raw_dm)
model_item_scores_DM_top_n_raw = get_LLM_value_per_item_top_n(raw_dm)

# Merge them on the grouping keys
model_item_scores_DM_raw = model_item_scores_DM_raw.merge(
    model_item_scores_DM_top_n_raw,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_DM_reversed = get_LLM_value_per_item(dm_reversed)
model_item_scores_DM_top_n_reversed = get_LLM_value_per_item_top_n(dm_reversed)

# Merge them on the grouping keys
model_item_scores_DM_reversed = model_item_scores_DM_reversed.merge(
    model_item_scores_DM_top_n_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_DM_raw_flipped = get_LLM_value_per_item(raw_dm_flipped)
model_item_scores_DM_top_n_raw_flipped = get_LLM_value_per_item_top_n(raw_dm_flipped)

# Merge them on the grouping keys
model_item_scores_DM_raw_flipped = model_item_scores_DM_raw_flipped.merge(
    model_item_scores_DM_top_n_raw_flipped,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_DM_flipped_reversed = get_LLM_value_per_item(dm_flipped_back_reversed_back)
model_item_scores_DM_top_n_flipped_reversed = get_LLM_value_per_item_top_n(dm_flipped_back_reversed_back)

# Merge them on the grouping keys
model_item_scores_DM_flipped_reversed = model_item_scores_DM_flipped_reversed.merge(
    model_item_scores_DM_top_n_flipped_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_DM_reversed], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_DM_raw], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, model_item_scores_DM_flipped_reversed], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, model_item_scores_DM_raw_flipped], ignore_index=True)

## DOSPERT SCALE

In [ ]:
# load data
DOSPERT_data = load_dataframes(task_name="DOSPERT", path = PATH)

In [ ]:
# normalise answer option sum to one
DOSPERT_data["prob_1"] = np.exp(DOSPERT_data["1"])/(np.exp(DOSPERT_data["1"]) + np.exp(DOSPERT_data["2"]) + np.exp(DOSPERT_data["3"]) + np.exp(DOSPERT_data["4"]) + np.exp(DOSPERT_data["5"]))
DOSPERT_data["prob_2"] = np.exp(DOSPERT_data["2"])/(np.exp(DOSPERT_data["1"]) + np.exp(DOSPERT_data["2"]) + np.exp(DOSPERT_data["3"]) + np.exp(DOSPERT_data["4"]) + np.exp(DOSPERT_data["5"]))
DOSPERT_data["prob_3"] = np.exp(DOSPERT_data["3"])/(np.exp(DOSPERT_data["1"]) + np.exp(DOSPERT_data["2"]) + np.exp(DOSPERT_data["3"]) + np.exp(DOSPERT_data["4"]) + np.exp(DOSPERT_data["5"]))
DOSPERT_data["prob_4"] = np.exp(DOSPERT_data["4"])/(np.exp(DOSPERT_data["1"]) + np.exp(DOSPERT_data["2"]) + np.exp(DOSPERT_data["3"]) + np.exp(DOSPERT_data["4"]) + np.exp(DOSPERT_data["5"]))
DOSPERT_data["prob_5"] = np.exp(DOSPERT_data["5"])/(np.exp(DOSPERT_data["1"]) + np.exp(DOSPERT_data["2"]) + np.exp(DOSPERT_data["3"]) + np.exp(DOSPERT_data["4"]) + np.exp(DOSPERT_data["5"]))


In [ ]:
# filter out probability LLM assigned to real item answer 
DOSPERT_data=filter_pred_prob(DOSPERT_data)

In [ ]:
# add whether item was reverse coded, none were!
# False for all
DOSPERT_data["reverse_coded"] = False

In [ ]:
# seperate the datasets
raw_dospert = DOSPERT_data[DOSPERT_data["flipped"] == "no"]
raw_dospert_flipped = DOSPERT_data[DOSPERT_data["flipped"] == "yes"]
dospert_flipped_back = raw_dospert_flipped.copy()
dospert_flipped_back["human_number"] = 6 - dospert_flipped_back["human_number"]

In [ ]:
# produce df with one value per model per item 
model_item_scores_DOSPERT_raw = get_LLM_value_per_item(raw_dospert)
model_item_scores_DOSPERT_top_n_raw = get_LLM_value_per_item_top_n(raw_dospert)

# Merge them on the grouping keys
model_item_scores_DOSPERT_raw = model_item_scores_DOSPERT_raw.merge(
    model_item_scores_DOSPERT_top_n_raw,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_DOSPERT_raw_flipped = get_LLM_value_per_item(raw_dospert_flipped)
model_item_scores_DOSPERT_top_n_raw_flipped = get_LLM_value_per_item_top_n(raw_dospert_flipped)

# Merge them on the grouping keys
model_item_scores_DOSPERT_raw_flipped = model_item_scores_DOSPERT_raw_flipped.merge(
    model_item_scores_DOSPERT_top_n_raw_flipped,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_DOSPERT_flipped_reversed = get_LLM_value_per_item(dospert_flipped_back)
model_item_scores_DOSPERT_top_n_flipped_reversed = get_LLM_value_per_item_top_n(dospert_flipped_back)

# Merge them on the grouping keys
model_item_scores_DOSPERT_flipped_reversed = model_item_scores_DOSPERT_flipped_reversed.merge(
    model_item_scores_DOSPERT_top_n_flipped_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

In [ ]:
# Adding task specific categories to save in all data

# add item categories
item_to_category = {
    1: "Social", 10: "Social", 16: "Social", 19: "Social", 23: "Social", 26: "Social", 34: "Social", 35: "Social",
    2: "Recreational", 6: "Recreational", 15: "Recreational", 17: "Recreational", 21: "Recreational", 31: "Recreational", 37: "Recreational", 38: "Recreational",
    3: "Gambling", 11: "Gambling", 22: "Gambling", 33: "Gambling",
    4: "Health", 8: "Health", 27: "Health", 29: "Health", 32: "Health", 36: "Health", 39: "Health", 40: "Health",
    5: "Ethical", 9: "Ethical", 12: "Ethical", 13: "Ethical", 14: "Ethical", 20: "Ethical", 25: "Ethical", 28: "Ethical",
    7: "Investment", 18: "Investment", 24: "Investment", 30: "Investment"
}

model_item_scores_DOSPERT_raw["category"] = model_item_scores_DOSPERT_raw["item"].map(item_to_category)
model_item_scores_DOSPERT_raw_flipped["category"] = model_item_scores_DOSPERT_raw_flipped["item"].map(item_to_category)
model_item_scores_DOSPERT_flipped_reversed["category"] = model_item_scores_DOSPERT_flipped_reversed["item"].map(item_to_category)


In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_DOSPERT_raw], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_DOSPERT_raw], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, model_item_scores_DOSPERT_flipped_reversed], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, model_item_scores_DOSPERT_raw_flipped], ignore_index=True)

## FTND SCALE

In [ ]:
# load data
FTND_data = load_dataframes(task_name="FTND", path = PATH)

In [ ]:
# normalise answer option sum to one (tun so als hätten wir sehr guten Prompt, dann würde LLM nur zwischen möglichen Antwortalternativen aussuchen, da simulieren wir dadurch)
mask = (FTND_data["item"] == 1)
FTND_data.loc[mask, "prob_1"] = np.exp(FTND_data.loc[mask, "1"])/(np.exp(FTND_data.loc[mask, "1"]) + np.exp(FTND_data.loc[mask, "2"]) + np.exp(FTND_data.loc[mask, "3"]))
FTND_data.loc[mask, "prob_2"] = np.exp(FTND_data.loc[mask, "2"])/(np.exp(FTND_data.loc[mask, "1"]) + np.exp(FTND_data.loc[mask, "2"]) + np.exp(FTND_data.loc[mask, "3"]))
FTND_data.loc[mask, "prob_3"] = np.exp(FTND_data.loc[mask, "3"])/(np.exp(FTND_data.loc[mask, "1"]) + np.exp(FTND_data.loc[mask, "2"]) + np.exp(FTND_data.loc[mask, "3"]))

mask = (FTND_data["item"].isin([3, 4, 6, 7]))
FTND_data.loc[mask, "prob_1"] = np.exp(FTND_data.loc[mask, "1"])/(np.exp(FTND_data.loc[mask, "1"]) + np.exp(FTND_data.loc[mask, "2"]))
FTND_data.loc[mask, "prob_2"] = np.exp(FTND_data.loc[mask, "2"])/(np.exp(FTND_data.loc[mask, "1"]) + np.exp(FTND_data.loc[mask, "2"]))

mask = (FTND_data["item"].isin([2, 5]))
FTND_data.loc[mask, "prob_1"] = np.exp(FTND_data.loc[mask, "1"])/(np.exp(FTND_data.loc[mask, "1"]) + np.exp(FTND_data.loc[mask, "2"]) + np.exp(FTND_data.loc[mask, "3"]) + np.exp(FTND_data.loc[mask, "4"]))
FTND_data.loc[mask, "prob_2"] = np.exp(FTND_data.loc[mask, "2"])/(np.exp(FTND_data.loc[mask, "1"]) + np.exp(FTND_data.loc[mask, "2"]) + np.exp(FTND_data.loc[mask, "3"]) + np.exp(FTND_data.loc[mask, "4"]))
FTND_data.loc[mask, "prob_3"] = np.exp(FTND_data.loc[mask, "3"])/(np.exp(FTND_data.loc[mask, "1"]) + np.exp(FTND_data.loc[mask, "2"]) + np.exp(FTND_data.loc[mask, "3"]) + np.exp(FTND_data.loc[mask, "4"]))
FTND_data.loc[mask, "prob_4"] = np.exp(FTND_data.loc[mask, "4"])/(np.exp(FTND_data.loc[mask, "1"]) + np.exp(FTND_data.loc[mask, "2"]) + np.exp(FTND_data.loc[mask, "3"]) + np.exp(FTND_data.loc[mask, "4"]))


In [ ]:
# filter out probability LLM assigned to real item answer 
FTND_data=filter_pred_prob(FTND_data)

In [ ]:
# add whether item was reverse coded
reverse_coded = {
    1: True, 2: True, 3: True,  4: True, 5: False,  6: True,  7: True
    }

# Apply mapping row-wise based on item number
FTND_data["reverse_coded"] = FTND_data["item"].map(reverse_coded)



In [ ]:
# seperate the datasets
raw_ftnd = FTND_data[FTND_data["flipped"] == False]
raw_ftnd_flipped = FTND_data[FTND_data["flipped"] == True]
ftnd_flipped_back = raw_ftnd_flipped.copy()
# flip back human answers where they were flipped
mask =(ftnd_flipped_back["item"] == 1)
ftnd_flipped_back.loc[mask, "human_number"] = 4 - ftnd_flipped_back.loc[mask, "human_number"]
mask =  (ftnd_flipped_back["item"].isin([3, 4, 6, 7]))
ftnd_flipped_back.loc[mask, "human_number"] = 3 - ftnd_flipped_back.loc[mask, "human_number"]
mask =  (ftnd_flipped_back["item"].isin([2, 5]))
ftnd_flipped_back.loc[mask, "human_number"] = 5 - ftnd_flipped_back.loc[mask, "human_number"]


In [ ]:
# Define mappings for each FTND question:
ftnd_maps = {
    1: {1: 2, 2: 0, 3: 1},             # FTND_1Eingangsfrage: smoke?
    2: {1: 3, 2: 2, 3: 1, 4: 0},       # FTND_1: time until first cigarette
    3: {1: 1, 2: 0},                   # FTND_2: difficult to refrain
    4: {1: 1, 2: 0},                   # FTND_3: which cigarette hardest to give up
    5: {1: 0, 2: 1, 3: 2, 4: 3},       # FTND_4: cigarettes per day
    6: {1: 1, 2: 0},                   # FTND_5: smoke more frequently in morning
    7: {1: 1, 2: 0}                    # FTND_6: smoke when ill
}

# Apply mapping row-wise based on item number
def recode_ftnd(row):
    mapping = ftnd_maps.get(row["item"])
    if mapping is not None:
        return mapping.get(row["human_number"], None) 
    return row["human_number"]  

# reverse LLM answers (again) where the items where reversed phrased
ftnd_reversed = raw_ftnd.copy()
ftnd_reversed["human_number"] = ftnd_reversed.apply(recode_ftnd, axis=1)
ftnd_flipped_back_reversed_back = ftnd_flipped_back.copy()
ftnd_flipped_back_reversed_back["human_number"] = ftnd_flipped_back_reversed_back.apply(recode_ftnd, axis=1)

In [ ]:
# produce df with one value per model per item 
model_item_scores_FTND_raw = get_LLM_value_per_item(raw_ftnd)
model_item_scores_FTND_top_n_raw = get_LLM_value_per_item_top_n(raw_ftnd)

# Merge them on the grouping keys
model_item_scores_FTND_raw = model_item_scores_FTND_raw.merge(
    model_item_scores_FTND_top_n_raw,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_FTND_reversed = get_LLM_value_per_item(ftnd_reversed)
model_item_scores_FTND_top_n_reversed = get_LLM_value_per_item_top_n(ftnd_reversed)

# Merge them on the grouping keys
model_item_scores_FTND_reversed = model_item_scores_FTND_reversed.merge(
    model_item_scores_FTND_top_n_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_FTND_raw_flipped = get_LLM_value_per_item(raw_ftnd_flipped)
model_item_scores_FTND_top_n_raw_flipped = get_LLM_value_per_item_top_n(raw_ftnd_flipped)

# Merge them on the grouping keys
model_item_scores_FTND_raw_flipped = model_item_scores_FTND_raw_flipped.merge(
    model_item_scores_FTND_top_n_raw_flipped,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_FTND_flipped_reversed = get_LLM_value_per_item(ftnd_flipped_back_reversed_back)
model_item_scores_FTND_top_n_flipped_reversed = get_LLM_value_per_item_top_n(ftnd_flipped_back_reversed_back)

# Merge them on the grouping keys
model_item_scores_FTND_flipped_reversed = model_item_scores_FTND_flipped_reversed.merge(
    model_item_scores_FTND_top_n_flipped_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_FTND_reversed], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_FTND_raw], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, model_item_scores_FTND_flipped_reversed], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, model_item_scores_FTND_raw_flipped], ignore_index=True)

## GABS SCALE

In [ ]:
# load data
GABS_data = load_dataframes(task_name="GABS", path = PATH)

In [ ]:
# normalise answer option sum to one

# columns representing log-probabilities
answer_cols = ["1", "2", "3", "4"]

# make a copy to avoid SettingWithCopy warnings
GABS_data = GABS_data.copy()

# case 1: item == 1 → only options 1 and 2
mask_item1 = GABS_data["item"] == 1
exp_vals_item1 = np.exp(GABS_data.loc[mask_item1, ["1", "2"]])
probs_item1 = exp_vals_item1.div(exp_vals_item1.sum(axis=1), axis=0)
probs_item1.columns = ["prob_1", "prob_2"]

# case 2: items 2–17 → options 1–4
mask_item2plus = GABS_data["item"].between(2, 17)
exp_vals_item2plus = np.exp(GABS_data.loc[mask_item2plus, answer_cols])
probs_item2plus = exp_vals_item2plus.div(exp_vals_item2plus.sum(axis=1), axis=0)
probs_item2plus.columns = [f"prob_{c}" for c in answer_cols]

# merge both parts back into original df
GABS_data = GABS_data.join(pd.concat([probs_item1, probs_item2plus]))



In [ ]:
# filter out probability LLM assigned to real item answer 
GABS_data=filter_pred_prob(GABS_data)

In [ ]:
# add whether item was reverse coded, none were!
# False for all
GABS_data["reverse_coded"] = False

In [ ]:
# seperate the datasets
raw_gabs = GABS_data[GABS_data["flipped"] == False]
raw_gabs_flipped = GABS_data[GABS_data["flipped"] == True]
gabs_flipped_back = raw_gabs_flipped.copy()
mask = (gabs_flipped_back["item"] == 0)
gabs_flipped_back.loc[mask, "human_number"] = 3 - gabs_flipped_back.loc[mask, "human_number"]
mask = (gabs_flipped_back["item"].isin(range(2,16)))
gabs_flipped_back.loc[mask, "human_number"] = 5 - gabs_flipped_back.loc[mask, "human_number"]

In [ ]:
# produce df with one value per model per item 
model_item_scores_GABS_raw = get_LLM_value_per_item(raw_gabs)
model_item_scores_GABS_top_n_raw = get_LLM_value_per_item_top_n(raw_gabs)

# Merge them on the grouping keys
model_item_scores_GABS_raw = model_item_scores_GABS_raw.merge(
    model_item_scores_GABS_top_n_raw,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_GABS_raw_flipped = get_LLM_value_per_item(raw_gabs_flipped)
model_item_scores_GABS_top_n_raw_flipped = get_LLM_value_per_item_top_n(raw_gabs_flipped)

# Merge them on the grouping keys
model_item_scores_GABS_raw_flipped = model_item_scores_GABS_raw_flipped.merge(
    model_item_scores_GABS_top_n_raw_flipped,
    on=["experiment", "model", "item"],
    how="inner" 
)


model_item_scores_GABS_flipped_reversed = get_LLM_value_per_item(gabs_flipped_back)
model_item_scores_GABS_top_n_flipped_reversed = get_LLM_value_per_item_top_n(gabs_flipped_back)

# Merge them on the grouping keys
model_item_scores_GABS_flipped_reversed = model_item_scores_GABS_flipped_reversed.merge(
    model_item_scores_GABS_top_n_flipped_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_GABS_raw], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_GABS_raw], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, model_item_scores_GABS_flipped_reversed], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, model_item_scores_GABS_raw_flipped], ignore_index=True)

## PG SCALE

In [ ]:
# load data
PG_data = load_dataframes(task_name="PG", path = PATH)

In [ ]:
# normalise answer option sum to one 

mask = (PG_data["item"].isin([1, 26]))
PG_data.loc[mask, "prob_1"] = np.exp(PG_data.loc[mask, "1"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]))
PG_data.loc[mask, "prob_2"] = np.exp(PG_data.loc[mask, "2"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]))



mask = (PG_data["item"].isin(range(2, 21)))
PG_data.loc[mask, "prob_1"] = np.exp(PG_data.loc[mask, "1"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]) + np.exp(PG_data.loc[mask, "3"]) + np.exp(PG_data.loc[mask, "4"]) + np.exp(PG_data.loc[mask, "5"]))
PG_data.loc[mask, "prob_2"] = np.exp(PG_data.loc[mask, "2"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]) + np.exp(PG_data.loc[mask, "3"]) + np.exp(PG_data.loc[mask, "4"]) + np.exp(PG_data.loc[mask, "5"]))
PG_data.loc[mask, "prob_3"] = np.exp(PG_data.loc[mask, "3"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]) + np.exp(PG_data.loc[mask, "3"]) + np.exp(PG_data.loc[mask, "4"]) + np.exp(PG_data.loc[mask, "5"]))
PG_data.loc[mask, "prob_4"] = np.exp(PG_data.loc[mask, "4"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]) + np.exp(PG_data.loc[mask, "3"]) + np.exp(PG_data.loc[mask, "4"]) + np.exp(PG_data.loc[mask, "5"]))
PG_data.loc[mask, "prob_5"] = np.exp(PG_data.loc[mask, "5"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]) + np.exp(PG_data.loc[mask, "3"]) + np.exp(PG_data.loc[mask, "4"]) + np.exp(PG_data.loc[mask, "5"]))



mask = (PG_data["item"] == 25)
PG_data.loc[mask, "prob_1"] = np.exp(PG_data.loc[mask, "1"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]) + np.exp(PG_data.loc[mask, "3"]) + np.exp(PG_data.loc[mask, "4"]) + np.exp(PG_data.loc[mask, "5"]) + np.exp(PG_data.loc[mask, "6"]))
PG_data.loc[mask, "prob_2"] = np.exp(PG_data.loc[mask, "2"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]) + np.exp(PG_data.loc[mask, "3"]) + np.exp(PG_data.loc[mask, "4"]) + np.exp(PG_data.loc[mask, "5"]) + np.exp(PG_data.loc[mask, "6"]))
PG_data.loc[mask, "prob_3"] = np.exp(PG_data.loc[mask, "3"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]) + np.exp(PG_data.loc[mask, "3"]) + np.exp(PG_data.loc[mask, "4"]) + np.exp(PG_data.loc[mask, "5"]) + np.exp(PG_data.loc[mask, "6"]))
PG_data.loc[mask, "prob_4"] = np.exp(PG_data.loc[mask, "4"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]) + np.exp(PG_data.loc[mask, "3"]) + np.exp(PG_data.loc[mask, "4"]) + np.exp(PG_data.loc[mask, "5"]) + np.exp(PG_data.loc[mask, "6"]))
PG_data.loc[mask, "prob_5"] = np.exp(PG_data.loc[mask, "5"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]) + np.exp(PG_data.loc[mask, "3"]) + np.exp(PG_data.loc[mask, "4"]) + np.exp(PG_data.loc[mask, "5"]) + np.exp(PG_data.loc[mask, "6"]))
PG_data.loc[mask, "prob_6"] = np.exp(PG_data.loc[mask, "6"])/(np.exp(PG_data.loc[mask, "1"]) + np.exp(PG_data.loc[mask, "2"]) + np.exp(PG_data.loc[mask, "3"]) + np.exp(PG_data.loc[mask, "4"]) + np.exp(PG_data.loc[mask, "5"]) + np.exp(PG_data.loc[mask, "6"]))


mask = (PG_data["item"].isin([21, 22, 23, 24, 27, 28, 29, 30, 31, 32]))
PG_data.loc[mask, "prob_0"] = np.exp(PG_data.loc[mask, "0"])/(np.exp(PG_data.loc[mask, "0"]) + np.exp(PG_data.loc[mask, "1"]))
PG_data.loc[mask, "prob_1"] = np.exp(PG_data.loc[mask, "1"])/(np.exp(PG_data.loc[mask, "0"]) + np.exp(PG_data.loc[mask, "1"]))


In [ ]:
# filter out probability LLM assigned to real item answer 
PG_data=filter_pred_prob(PG_data)

In [ ]:
# add whether item was reverse coded, all were!
# True for all
PG_data["reverse_coded"] = True

In [ ]:
# seperate the datasets
raw_pg = PG_data[PG_data["flipped"] == False]
raw_pg_flipped = PG_data[PG_data["flipped"] == True]
pg_flipped_back = raw_pg_flipped.copy()

# flip back human answers where they were flipped
mask = (pg_flipped_back["item"].isin([1, 26]))
pg_flipped_back.loc[mask, "human_number"] = 3 - pg_flipped_back.loc[mask, "human_number"]

mask = (pg_flipped_back["item"].isin(range(2, 21)))
pg_flipped_back.loc[mask, "human_number"] = 6 - pg_flipped_back.loc[mask, "human_number"]

mask = (pg_flipped_back["item"] == 25)
pg_flipped_back.loc[mask, "human_number"] = 7 - pg_flipped_back.loc[mask, "human_number"]

mask = (pg_flipped_back["item"].isin([21, 22, 23, 24, 27, 28, 29, 30, 31, 32]))
pg_flipped_back.loc[mask, "human_number"] = 1 - pg_flipped_back.loc[mask, "human_number"]


In [ ]:
# Define mappings for each GABS question:
pg_maps = {     
    1: {1: 1, 2: 0},                      
    2: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    3: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    4: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    5: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    6: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    7: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    8: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    9: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    10: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    11: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    12: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    13: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    14: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    15: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    16: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    17: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    18: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    19: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    20: {1: 4, 2: 3, 3: 2, 4: 1, 5: 0},
    26: {1: 1, 2: 0}
}

# Apply mapping row-wise based on item number
def recode_pg(row):
    mapping = pg_maps.get(row["item"])
    if mapping is not None:
        return mapping.get(row["human_number"], None)  # None if invalid code
    return row["human_number"]  


pg_reversed = raw_pg.copy()
pg_reversed["human_number"] = pg_reversed.apply(recode_pg, axis=1)

pg_flipped_back_reversed_back = pg_flipped_back.copy()
pg_flipped_back_reversed_back["human_number"] = pg_flipped_back_reversed_back.apply(recode_pg, axis=1)

# jetzt ist es konsistent mit Freys df quest_proc (außer an den Items, wo es in der gleichen Skale bei ESS_GABS_ausserh_01-10 plötzlich ?vergessen? wurde bei Frey)
# aber meiner Meinung nach müsste man, damit man die Skala in binned factors umwandeln kann, noch alles in die gleiche Richtung bringen,
# 1 und 26 sind in falscher Richtung! -> habe ich jetzt gefixt obwohl Abweichung

In [ ]:
# produce df with one value per model per item 
model_item_scores_PG_raw = get_LLM_value_per_item(raw_pg)
model_item_scores_PG_top_n_raw = get_LLM_value_per_item_top_n(raw_pg)

# Merge them on the grouping keys
model_item_scores_PG_raw = model_item_scores_PG_raw.merge(
    model_item_scores_PG_top_n_raw,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_PG_reversed = get_LLM_value_per_item(pg_reversed)
model_item_scores_PG_top_n_reversed = get_LLM_value_per_item_top_n(pg_reversed)

# Merge them on the grouping keys
model_item_scores_PG_reversed = model_item_scores_PG_reversed.merge(
    model_item_scores_PG_top_n_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_PG_raw_flipped = get_LLM_value_per_item(raw_pg_flipped)
model_item_scores_PG_top_n_raw_flipped = get_LLM_value_per_item_top_n(raw_pg_flipped)

# Merge them on the grouping keys
model_item_scores_PG_raw_flipped = model_item_scores_PG_raw_flipped.merge(
    model_item_scores_PG_top_n_raw_flipped,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_PG_flipped_reversed = get_LLM_value_per_item(pg_flipped_back_reversed_back)
model_item_scores_PG_top_n_flipped_reversed = get_LLM_value_per_item_top_n(pg_flipped_back_reversed_back)

# Merge them on the grouping keys
model_item_scores_PG_flipped_reversed = model_item_scores_PG_flipped_reversed.merge(
    model_item_scores_PG_top_n_flipped_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_PG_reversed], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_PG_raw], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, model_item_scores_PG_flipped_reversed], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, model_item_scores_PG_raw_flipped], ignore_index=True)

## PRI SCALE

In [ ]:
# load data
PRI_data = load_dataframes(task_name="PRI", path = PATH)

In [ ]:
# add whether item was reverse coded, none were!
# False for all
PRI_data["reverse_coded"] = False

In [ ]:
# normalise answer option sum to one 

mask = (PRI_data["item"].isin([1, 3, 5, 7, 9, 11, 13, 15]))
PRI_data.loc[mask, "prob_1"] = np.exp(PRI_data.loc[mask, "1"])/(np.exp(PRI_data.loc[mask, "1"]) + np.exp(PRI_data.loc[mask, "2"]))
PRI_data.loc[mask, "prob_2"] = np.exp(PRI_data.loc[mask, "2"])/(np.exp(PRI_data.loc[mask, "1"]) + np.exp(PRI_data.loc[mask, "2"]))


mask = (PRI_data["item"].isin([2, 4, 6, 8, 10, 12, 14, 16]))
PRI_data.loc[mask, "prob_1"] = np.exp(PRI_data.loc[mask, "1"])/(np.exp(PRI_data.loc[mask, "1"]) + np.exp(PRI_data.loc[mask, "2"]) + np.exp(PRI_data.loc[mask, "3"]) + np.exp(PRI_data.loc[mask, "4"]) + np.exp(PRI_data.loc[mask, "5"]) + np.exp(PRI_data.loc[mask, "6"]) + np.exp(PRI_data.loc[mask, "7"]))
PRI_data.loc[mask, "prob_2"] = np.exp(PRI_data.loc[mask, "2"])/(np.exp(PRI_data.loc[mask, "1"]) + np.exp(PRI_data.loc[mask, "2"]) + np.exp(PRI_data.loc[mask, "3"]) + np.exp(PRI_data.loc[mask, "4"]) + np.exp(PRI_data.loc[mask, "5"]) + np.exp(PRI_data.loc[mask, "6"]) + np.exp(PRI_data.loc[mask, "7"]))
PRI_data.loc[mask, "prob_3"] = np.exp(PRI_data.loc[mask, "3"])/(np.exp(PRI_data.loc[mask, "1"]) + np.exp(PRI_data.loc[mask, "2"]) + np.exp(PRI_data.loc[mask, "3"]) + np.exp(PRI_data.loc[mask, "4"]) + np.exp(PRI_data.loc[mask, "5"]) + np.exp(PRI_data.loc[mask, "6"]) + np.exp(PRI_data.loc[mask, "7"]))
PRI_data.loc[mask, "prob_4"] = np.exp(PRI_data.loc[mask, "4"])/(np.exp(PRI_data.loc[mask, "1"]) + np.exp(PRI_data.loc[mask, "2"]) + np.exp(PRI_data.loc[mask, "3"]) + np.exp(PRI_data.loc[mask, "4"]) + np.exp(PRI_data.loc[mask, "5"]) + np.exp(PRI_data.loc[mask, "6"]) + np.exp(PRI_data.loc[mask, "7"]))
PRI_data.loc[mask, "prob_5"] = np.exp(PRI_data.loc[mask, "5"])/(np.exp(PRI_data.loc[mask, "1"]) + np.exp(PRI_data.loc[mask, "2"]) + np.exp(PRI_data.loc[mask, "3"]) + np.exp(PRI_data.loc[mask, "4"]) + np.exp(PRI_data.loc[mask, "5"]) + np.exp(PRI_data.loc[mask, "6"]) + np.exp(PRI_data.loc[mask, "7"]))
PRI_data.loc[mask, "prob_6"] = np.exp(PRI_data.loc[mask, "6"])/(np.exp(PRI_data.loc[mask, "1"]) + np.exp(PRI_data.loc[mask, "2"]) + np.exp(PRI_data.loc[mask, "3"]) + np.exp(PRI_data.loc[mask, "4"]) + np.exp(PRI_data.loc[mask, "5"]) + np.exp(PRI_data.loc[mask, "6"]) + np.exp(PRI_data.loc[mask, "7"]))
PRI_data.loc[mask, "prob_7"] = np.exp(PRI_data.loc[mask, "7"])/(np.exp(PRI_data.loc[mask, "1"]) + np.exp(PRI_data.loc[mask, "2"]) + np.exp(PRI_data.loc[mask, "3"]) + np.exp(PRI_data.loc[mask, "4"]) + np.exp(PRI_data.loc[mask, "5"]) + np.exp(PRI_data.loc[mask, "6"]) + np.exp(PRI_data.loc[mask, "7"]))

In [ ]:
# filter out probability LLM assigned to real item answer 
PRI_data=filter_pred_prob(PRI_data)

In [ ]:
# seperate the datasets
raw_pri = PRI_data[PRI_data["flipped"] == False]
raw_pri_flipped = PRI_data[PRI_data["flipped"] == True]
pri_flipped_back = raw_pri_flipped.copy()
# flip back human answers where they were flipped
mask = (pri_flipped_back["item"].isin([1, 3, 5, 7, 9, 11, 13, 15]))
pri_flipped_back.loc[mask, "human_number"] = 3 - pri_flipped_back.loc[mask, "human_number"]

mask =  (pri_flipped_back["item"].isin([2, 4, 6, 8, 10, 12, 14, 16]))
pri_flipped_back.loc[mask, "human_number"] = 8 - pri_flipped_back.loc[mask, "human_number"]

In [ ]:
# produce df with one value per model per item 
model_item_scores_PRI_raw = get_LLM_value_per_item(raw_pri)
model_item_scores_PRI_top_n_raw = get_LLM_value_per_item_top_n(raw_pri)

# Merge them on the grouping keys
model_item_scores_PRI_raw = model_item_scores_PRI_raw.merge(
    model_item_scores_PRI_top_n_raw,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_PRI_raw_flipped = get_LLM_value_per_item(raw_pri_flipped)
model_item_scores_PRI_top_n_raw_flipped = get_LLM_value_per_item_top_n(raw_pri_flipped)

# Merge them on the grouping keys
model_item_scores_PRI_raw_flipped = model_item_scores_PRI_raw_flipped.merge(
    model_item_scores_PRI_top_n_raw_flipped,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_PRI_flipped_reversed = get_LLM_value_per_item(pri_flipped_back)
model_item_scores_PRI_top_n_flipped_reversed = get_LLM_value_per_item_top_n(pri_flipped_back)

# Merge them on the grouping keys
model_item_scores_PRI_flipped_reversed = model_item_scores_PRI_flipped_reversed.merge(
    model_item_scores_PRI_top_n_flipped_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

In [ ]:
# Adding task specific categories to save in all data

# add item categories
item_to_category = {
     1: "decision", 3: "decision", 5: "decision", 7: "decision", 9: "decision", 11: "decision", 13: "decision", 15: "decision",
     2: "certainty", 4: "certainty", 6: "certainty", 8: "certainty", 10: "certainty", 12: "certainty", 14: "certainty", 16: "certainty"
}

model_item_scores_PRI_raw["category"] = model_item_scores_PRI_raw["item"].map(item_to_category)
model_item_scores_PRI_raw_flipped["category"] = model_item_scores_PRI_raw_flipped["item"].map(item_to_category)
model_item_scores_PRI_flipped_reversed["category"] = model_item_scores_PRI_flipped_reversed["item"].map(item_to_category)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_PRI_raw], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_PRI_raw], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, model_item_scores_PRI_flipped_reversed], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, model_item_scores_PRI_raw_flipped], ignore_index=True)

## SOEP SCALE

In [ ]:
# load data
SOEP_data = load_dataframes(task_name="SOEP", path = PATH)

In [ ]:
# get probabilities out of log-probabilities

cols = [str(i) for i in range(1, 12)]
# Compute normalized probabilities
exp_vals = np.exp(SOEP_data[cols])
prob_vals = exp_vals.div(exp_vals.sum(axis=1), axis=0)

# Rename columns all at once
prob_vals.columns = [f"prob_{i}" for i in range(1, 12)]

# Join to original dataframe in one step
SOEP_data = pd.concat([SOEP_data, prob_vals], axis=1).copy()

In [ ]:
# add whether item was reverse coded, none were!
# False for all
SOEP_data["reverse_coded"] = False

In [ ]:
# filter out probability LLM assigned to real item answer 
SOEP_data=filter_pred_prob(SOEP_data)

In [ ]:
# seperate the datasets
raw_soep = SOEP_data[SOEP_data["flipped"] == "no"]
raw_soep_flipped = SOEP_data[SOEP_data["flipped"] == "yes"]
soep_flipped_back = raw_soep_flipped.copy()
soep_flipped_back["human_number"] = 12 - soep_flipped_back["human_number"]

In [ ]:
# produce df with one value per model per item 
model_item_scores_SOEP_raw = get_LLM_value_per_item(raw_soep)
model_item_scores_SOEP_top_n_raw = get_LLM_value_per_item_top_n(raw_soep)

# Merge them on the grouping keys
model_item_scores_SOEP_raw = model_item_scores_SOEP_raw.merge(
    model_item_scores_SOEP_top_n_raw,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_SOEP_raw_flipped = get_LLM_value_per_item(raw_pri_flipped)
model_item_scores_SOEP_top_n_raw_flipped = get_LLM_value_per_item_top_n(raw_pri_flipped)

# Merge them on the grouping keys
model_item_scores_SOEP_raw_flipped = model_item_scores_SOEP_raw_flipped.merge(
    model_item_scores_SOEP_top_n_raw_flipped,
    on=["experiment", "model", "item"],
    how="inner" 
)


model_item_scores_SOEP_flipped_reversed = get_LLM_value_per_item(soep_flipped_back)
model_item_scores_SOEP_top_n_flipped_reversed = get_LLM_value_per_item_top_n(soep_flipped_back)

# Merge them on the grouping keys
model_item_scores_SOEP_flipped_reversed = model_item_scores_SOEP_flipped_reversed.merge(
    model_item_scores_SOEP_top_n_flipped_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)


In [ ]:
# Adding task specific categories to save in all data

# add item categories
item_to_category = {
     1: "SOEP", 2: "SOEPdri", 3: "SOEPfin",  4: "SOEPrec", 5: "SOEPocc",  6: "SOEPhea",  7: "SOEPsoc"
}

model_item_scores_SOEP_raw["category"] = model_item_scores_SOEP_raw["item"].map(item_to_category)
model_item_scores_SOEP_raw_flipped["category"] = model_item_scores_SOEP_raw_flipped["item"].map(item_to_category)
model_item_scores_SOEP_flipped_reversed["category"] = model_item_scores_SOEP_flipped_reversed["item"].map(item_to_category)


In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_SOEP_raw], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_SOEP_raw], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, model_item_scores_SOEP_flipped_reversed], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, model_item_scores_SOEP_raw_flipped], ignore_index=True)

## SSSV SCALE

In [ ]:
# load data
SSSV_data = load_dataframes(task_name="SSSV", path = PATH)

In [ ]:
# normalise answer option sum to one
SSSV_data["prob_1"] = np.exp(SSSV_data["1"])/(np.exp(SSSV_data["1"]) + np.exp(SSSV_data["2"]))
SSSV_data["prob_2"] = np.exp(SSSV_data["2"])/(np.exp(SSSV_data["1"]) + np.exp(SSSV_data["2"]))

In [ ]:
# filter out probability LLM assigned to real item answer 
SSSV_data=filter_pred_prob(SSSV_data)

In [ ]:
# add item categories
item_to_category = {
     3: "SStas", 11: "SStas", 16: "SStas", 17: "SStas", 20: "SStas", 21: "SStas", 23: "SStas", 28: "SStas", 38: "SStas", 40: "SStas",
     4: "SSexp", 6: "SSexp", 9: "SSexp", 10: "SSexp", 14: "SSexp", 18: "SSexp", 19: "SSexp", 22: "SSexp", 26: "SSexp", 37: "SSexp",
     1: "SSdis", 12: "SSdis", 13: "SSdis", 25: "SSdis", 29: "SSdis", 30: "SSdis", 32: "SSdis", 33: "SSdis", 35: "SSdis", 36: "SSdis",
     2: "SSbor", 5: "SSbor", 7: "SSbor", 8: "SSbor", 15: "SSbor", 24: "SSbor", 27: "SSbor", 31: "SSbor", 34: "SSbor", 39: "SSbor"
}

SSSV_data["category"] = SSSV_data["item"].map(item_to_category)

In [ ]:
# add whether item was reverse coded
reverse_coded = {
     1: True, 2: False, 3: True, 4: False, 5: True, 6: True, 7: False, 8: True, 9: True, 10: False, 
     11: False, 12: False, 13: False, 14: True, 15: False, 16: True, 17: True, 18: True, 19: False, 20: False,
     21: False, 22: True, 23: True, 24: True, 25: False, 26: False, 27: False, 28: True, 29: True, 30: False,
     31: False, 32: True, 33: False, 34: True, 35: False, 36: True, 37: False, 38: False, 39: True, 40: False

}

# Apply mapping row-wise based on item number
SSSV_data["reverse_coded"] = SSSV_data["item"].map(reverse_coded)

In [ ]:
# seperate the datasets
raw_sssv = SSSV_data[SSSV_data["flipped"] == False]
raw_sssv_flipped = SSSV_data[SSSV_data["flipped"] == True]
sssv_flipped_back = raw_sssv_flipped.copy()
sssv_flipped_back["human_number"] = 3 - sssv_flipped_back["human_number"]

In [ ]:
# reverse human answers (again) where the items where reversed phrased

sssv_reversed = raw_sssv.copy()
# flip back answers that where reverse coded
mask = (sssv_reversed["reverse_coded"] == True)
sssv_reversed.loc[mask, "human_number"] = 3 - sssv_reversed.loc[mask, "human_number"]

sssv_flipped_back_reversed_back = sssv_flipped_back.copy()
# flip back answers that where reverse coded
mask = (sssv_flipped_back_reversed_back["reverse_coded"] == True)
sssv_flipped_back_reversed_back.loc[mask, "human_number"] = 3 - sssv_flipped_back_reversed_back.loc[mask, "human_number"]


In [ ]:
# produce df with one value per model per item 
model_item_scores_SSSV_raw = get_LLM_value_per_item(raw_sssv)
model_item_scores_SSSV_top_n_raw = get_LLM_value_per_item_top_n(raw_sssv)

# Merge them on the grouping keys
model_item_scores_SSSV_raw = model_item_scores_SSSV_raw.merge(
    model_item_scores_SSSV_top_n_raw,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_SSSV_reversed = get_LLM_value_per_item(sssv_reversed)
model_item_scores_SSSV_top_n_reversed = get_LLM_value_per_item_top_n(sssv_reversed)

# Merge them on the grouping keys
model_item_scores_SSSV_reversed = model_item_scores_SSSV_reversed.merge(
    model_item_scores_SSSV_top_n_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_SSSV_raw_flipped = get_LLM_value_per_item(raw_sssv_flipped)
model_item_scores_SSSV_top_n_raw_flipped = get_LLM_value_per_item_top_n(raw_sssv_flipped)

# Merge them on the grouping keys
model_item_scores_SSSV_raw_flipped = model_item_scores_SSSV_raw_flipped.merge(
    model_item_scores_SSSV_top_n_raw_flipped,
    on=["experiment", "model", "item"],
    how="inner" 
)

model_item_scores_SSSV_flipped_reversed = get_LLM_value_per_item(sssv_flipped_back_reversed_back)
model_item_scores_SSSV_top_n_flipped_reversed = get_LLM_value_per_item_top_n(sssv_flipped_back_reversed_back)

# Merge them on the grouping keys
model_item_scores_SSSV_flipped_reversed = model_item_scores_SSSV_flipped_reversed.merge(
    model_item_scores_SSSV_top_n_flipped_reversed,
    on=["experiment", "model", "item"],
    how="inner" 
)

In [ ]:
# Adding task specific categories to save in all data

# add item categories
item_to_category = {
     3: "SStas", 11: "SStas", 16: "SStas", 17: "SStas", 20: "SStas", 21: "SStas", 23: "SStas", 28: "SStas", 38: "SStas", 40: "SStas",
     4: "SSexp", 6: "SSexp", 9: "SSexp", 10: "SSexp", 14: "SSexp", 18: "SSexp", 19: "SSexp", 22: "SSexp", 26: "SSexp", 37: "SSexp",
     1: "SSdis", 12: "SSdis", 13: "SSdis", 25: "SSdis", 29: "SSdis", 30: "SSdis", 32: "SSdis", 33: "SSdis", 35: "SSdis", 36: "SSdis",
     2: "SSbor", 5: "SSbor", 7: "SSbor", 8: "SSbor", 15: "SSbor", 24: "SSbor", 27: "SSbor", 31: "SSbor", 34: "SSbor", 39: "SSbor"
}

model_item_scores_SSSV_raw["category"] = model_item_scores_SSSV_raw["item"].map(item_to_category)
model_item_scores_SSSV_reversed["category"] = model_item_scores_SSSV_reversed["item"].map(item_to_category)
model_item_scores_SSSV_raw_flipped["category"] = model_item_scores_SSSV_raw_flipped["item"].map(item_to_category)
model_item_scores_SSSV_flipped_reversed["category"] = model_item_scores_SSSV_flipped_reversed["item"].map(item_to_category)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_SSSV_reversed], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_SSSV_raw], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, model_item_scores_SSSV_flipped_reversed], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, model_item_scores_SSSV_raw_flipped], ignore_index=True)

## Behavioural Tasks 

In [ ]:
# Helpers

# filter out probability LLM assigned to real item answer  ------------------------------------------
def filter_pred_prob_including_key_logic(data, key1_column="box_1_key", key2_column= "box_2_key"):
    # Determine whether the human decision matches box_1 or box_2
    mask_box1 = data["human_decision"] == data[key1_column]
    mask_box2 = data["human_decision"] == data[key2_column]

    # Assign probability based on which box matches
    data["prob_pred"] = data["prob_1"].where(mask_box1, data["prob_2"].where(mask_box2, None))
    
    return data



# Funktion for top-n weighted score
def compute_top_n_weighted_score(group, n = 100, human_col = "decision_num"):
    # sort rows by probabillities, descending
    top_n = group.sort_values("prob_pred", ascending=False).head(n)
    # Numerator = Summe von (Antwort * Wahrscheinlichkeit)
    numerator = (top_n[human_col] * top_n["prob_pred"]).sum()
    # Denominator = Summe von Wahrscheinlichkeiten der Top n
    denominator = top_n["prob_pred"].sum()
    return numerator / denominator if denominator > 0 else None

# produce df with one value per model per item for top n version --------------------------------------------------
def get_LLM_value_per_item_top_n(data, human_col = "decision_num", item_name = "round"):
    new_df = (
    data.groupby(["experiment", "model", item_name])[[human_col, "prob_pred"]]
      .apply(lambda group: compute_top_n_weighted_score(group, human_col=human_col))
      .reset_index(name="score_top_n")
    )
    return(new_df)


### BART TASK

In [ ]:
# load data
BART_data = load_dataframes(task_name="BART", path = PATH)

In [ ]:
# normalise answer option sum to one
BART_data["prob_pump"] = np.exp(BART_data["log_prob_pump"])/(np.exp(BART_data["log_prob_pump"]) + np.exp(BART_data["log_prob_stop"]))
BART_data["prob_stop"] = np.exp(BART_data["log_prob_stop"])/(np.exp(BART_data["log_prob_pump"]) + np.exp(BART_data["log_prob_stop"]))

In [ ]:
# filter out probability LLM assigned to real item answer 
BART_data=filter_pred_prob(BART_data, human_col = "human_decision")

In [ ]:
# per balloon calc avg probabiility for actual human decision 
avg_prob = BART_data.groupby(["experiment", "model", "participant", "round"])["prob_pred"].mean().reset_index()

In [ ]:
# per balloon extract number of pumps
max_decision = BART_data.groupby(["experiment", "model", "participant", "round"])["decision_num"].max().reset_index()

In [ ]:
# merge the two aggregated numbers (pred_prob and decision_num)per person in one df
summary = pd.merge(avg_prob, max_decision, on=["experiment", "model", "participant", "round"])

In [ ]:
# per balloon the probability times the number of pumps, normalise over people (to have one prob per model)
def get_LLM_value_per_item(data):
    grouped = data.groupby(["experiment", "model", "round"])
    score = (grouped["decision_num"].apply(lambda x: (x * data.loc[x.index, "prob_pred"]).sum())
             / grouped["prob_pred"].sum())
    return score.reset_index(name="score")


model_item_scores_BART = get_LLM_value_per_item(summary)
model_item_scores_BART_top_n = get_LLM_value_per_item_top_n(summary)

# Merge them on the grouping keys
model_item_scores_BART = model_item_scores_BART.merge(
    model_item_scores_BART_top_n,
    on=["experiment", "model", "round"],
    how="inner" 
)


In [ ]:
# rename BART round column to "item" to fit to all_data
model_item_scores_BART = model_item_scores_BART.rename(columns = {"round": "item"})

In [ ]:
# organize outcome dfs
model_item_scores_BART["reverse_coded"] = False

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_BART], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_BART], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
# no flipped data

## CCT TASK

In [ ]:
# load data
CCT_data = load_dataframes(task_name="CCT", path = PATH)

In [ ]:
# normalise answer option sum to one
CCT_data["prob_flip"] = np.exp(CCT_data["log_prob_flip"])/(np.exp(CCT_data["log_prob_flip"]) + np.exp(CCT_data["log_prob_stop"]))
CCT_data["prob_stop"] = np.exp(CCT_data["log_prob_stop"])/(np.exp(CCT_data["log_prob_flip"]) + np.exp(CCT_data["log_prob_stop"]))

In [ ]:
# filter out probability LLM assigned to real item answer 
CCT_data=filter_pred_prob(CCT_data, human_col = "human_decision")

In [ ]:
# per round calc the average probability for the actual human decision (then we have one probability per model per round)
avg_prob_CCT = CCT_data.groupby(["experiment", "model", "participant", "round"])["prob_pred"].mean().reset_index()

In [ ]:
# per round extract number of decisions
max_decision_CCT = CCT_data.groupby(["experiment", "model", "participant", "round"])["decision_num"].max().reset_index()


In [ ]:
# merge the two aggregated numbers (pred_prob and decision_num)per person in one df
summary_CCT = pd.merge(avg_prob_CCT, max_decision_CCT, on=["experiment", "model", "participant", "round"])


In [ ]:
# per round Probability times decision_num, normalise over humans 
model_item_scores_CCT = get_LLM_value_per_item(summary_CCT)

model_item_scores_CCT_top_n = get_LLM_value_per_item_top_n(summary_CCT)

# Merge them on the grouping keys
model_item_scores_CCT = model_item_scores_CCT.merge(
    model_item_scores_CCT_top_n,
    on=["experiment", "model", "round"],
    how="inner" 
)

In [ ]:
# rename CCT round column to "item" to fit to all_data
model_item_scores_CCT = model_item_scores_CCT.rename(columns = {"round": "item"})

In [ ]:
# organize outcome dfs
model_item_scores_CCT["reverse_coded"] = False

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_CCT], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_CCT], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
# no flipped data

## DFD TASK

In [ ]:
# load data
DFD_data = load_dataframes(task_name="DFD", path = PATH)
DFD_human_data = pd.read_csv("../../data/raw/risk_data/orig_human_data/dfd_perprob.csv")

In [ ]:
# normalise answer option sum to one
DFD_data["prob_1"] = np.exp(DFD_data["log_prob_box_1"])/(np.exp(DFD_data["log_prob_box_1"]) + np.exp(DFD_data["log_prob_box_2"]))
DFD_data["prob_2"] = np.exp(DFD_data["log_prob_box_2"])/(np.exp(DFD_data["log_prob_box_1"]) + np.exp(DFD_data["log_prob_box_2"]))

In [ ]:
# filter out probability LLM assigned to real item answer 
DFD_data=filter_pred_prob_including_key_logic(DFD_data)

In [ ]:
# Merge only selected columns from DFD_human_data
DFD_data = DFD_data.merge(
    DFD_human_data[["partid", "gamble_ind", "gamble_lab", "H", "R"]],
    left_on=["participant", "round"],
    right_on=["partid", "gamble_ind"],
    how="left"
)

# Drop duplicate key columns
DFD_data = DFD_data.drop(columns=["partid", "gamble_ind"])


In [ ]:
# produce df with one value per model per item -------------------------------------------------------
def get_LLM_value_per_item(data):
    grouped = data.groupby(["experiment", "model", "gamble_lab"])
    prob_sum = grouped["prob_pred"].sum()
    
    # compute weighted means
    scoreH = (grouped["H"].apply(lambda x: (x * data.loc[x.index, "prob_pred"]).sum()) / prob_sum)
    scoreR = (grouped["R"].apply(lambda x: (x * data.loc[x.index, "prob_pred"]).sum()) / prob_sum)
    
    # combine into one DataFrame
    result = pd.concat([scoreH, scoreR], axis=1).reset_index()
    result.columns = ["experiment", "model", "gamble_lab", "H_score", "score"]
    
    return result


model_item_scores_DFD = get_LLM_value_per_item(DFD_data)


model_item_scores_DFD_top_n = get_LLM_value_per_item_top_n(DFD_data, human_col="R", item_name="gamble_lab")

# Merge them on the grouping keys
model_item_scores_DFD = model_item_scores_DFD.merge(
    model_item_scores_DFD_top_n,
    on=["experiment", "model", "gamble_lab"],
    how="inner" 
)

In [ ]:
# rename DFD gamble_lab column to "item" to fit to all_data
model_item_scores_DFD = model_item_scores_DFD.rename(columns = {"gamble_lab": "item"})

In [ ]:
# add which item was reverse coded
risky_decision_table = {
    "he04_1": False,
    "he04_2": False,
    "he04_3": True,
    "he04_4": True,
    "he04_5": False,
    "he04_6": False,
    "he04_2inv": True,
    "he04_6inv": True
}

model_item_scores_DFD["reverse_coded"] = model_item_scores_DFD["item"].map(risky_decision_table)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_DFD], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_DFD], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
# no flipped data

## DFE TASK

In [ ]:
# load data
DFE_data = load_dataframes(task_name="DFE", path = PATH)
DFE_human_data = pd.read_csv("../../data/raw/risk_data/orig_human_data/dfe_perprob.csv")

In [ ]:
# normalise answer option sum to one
DFE_data["prob_1"] = np.exp(DFE_data["log_prob_box_1"])/(np.exp(DFE_data["log_prob_box_1"]) + np.exp(DFE_data["log_prob_box_2"]))
DFE_data["prob_2"] = np.exp(DFE_data["log_prob_box_2"])/(np.exp(DFE_data["log_prob_box_1"]) + np.exp(DFE_data["log_prob_box_2"]))

In [ ]:
# filter out probability LLM assigned to real item answer 
DFE_data=filter_pred_prob_including_key_logic(DFE_data)

In [ ]:
# Merge only selected columns from DFD_human_data
DFE_data = DFE_data.merge(
    DFE_human_data[["partid", "gamble_ind", "gamble_lab", "R", "Rexp"]],
    left_on=["participant", "round"],
    right_on=["partid", "gamble_ind"],
    how="left"
)

# Drop duplicate key columns
DFE_data = DFE_data.drop(columns=["partid", "gamble_ind"])


In [ ]:
# produce df with one value per model per item -------------------------------------------------------
def get_LLM_value_per_item(data):
    grouped = data.groupby(["experiment", "model", "gamble_lab"])
    prob_sum = grouped["prob_pred"].sum()
    
    # compute weighted means
    scoreR = (grouped["R"].apply(lambda x: (x * data.loc[x.index, "prob_pred"]).sum()) / prob_sum)
    scoreRexp = (grouped["Rexp"].apply(lambda x: (x * data.loc[x.index, "prob_pred"]).sum()) / prob_sum)
    
    # combine into one DataFrame
    result = pd.concat([scoreR, scoreRexp], axis=1).reset_index()
    result.columns = ["experiment", "model", "gamble_lab", "score_expected", "score"]
    
    return result


model_item_scores_DFE = get_LLM_value_per_item(DFE_data)

model_item_scores_DFE_top_n = get_LLM_value_per_item_top_n(DFE_data, human_col="Rexp", item_name="gamble_lab")

# Merge them on the grouping keys
model_item_scores_DFE = model_item_scores_DFE.merge(
    model_item_scores_DFE_top_n,
    on=["experiment", "model", "gamble_lab"],
    how="inner" 
)

In [ ]:
# rename DFE gamble_lab column to "item" to fit to all_data
model_item_scores_DFE = model_item_scores_DFE.rename(columns = {"gamble_lab": "item"})

In [ ]:
# add which item was reverse coded
risky_decision_table = {
    "he04_1": False,
    "he04_2": False,
    "he04_3": True,
    "he04_4": True,
    "he04_5": False,
    "he04_6": False,
    "he04_2inv": True,
    "he04_6inv": True
}
# evtl falsch, weil experienced risk nicht so einfach vergleichbar, aber da ich eh nichts umdrehe, egal.
model_item_scores_DFE["reverse_coded"] = model_item_scores_DFE["item"].map(risky_decision_table)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_DFE], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_DFE], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
# no flipped data

## LOT TASK

In [ ]:
# load data
LOT_data = load_dataframes(task_name="LOT", path = PATH)
LOT_human_data = pd.read_csv("../../data/raw/risk_data/orig_human_data/lotteries.csv")

In [ ]:
# normalise answer option sum to one
LOT_data["prob_1"] = np.exp(LOT_data["log_prob_box_1"])/(np.exp(LOT_data["log_prob_box_1"]) + np.exp(LOT_data["log_prob_box_2"]))
LOT_data["prob_2"] = np.exp(LOT_data["log_prob_box_2"])/(np.exp(LOT_data["log_prob_box_1"]) + np.exp(LOT_data["log_prob_box_2"]))

In [ ]:
# filter out probability LLM assigned to real item answer 
LOT_data=filter_pred_prob_including_key_logic(LOT_data)

In [ ]:
# Merge only selected columns from DFD_human_data
LOT_data = LOT_data.merge(
    LOT_human_data[["partid", "Dec_ID", "R"]],
    left_on=["participant", "round"],
    right_on=["partid", "Dec_ID"],
    how="left"
)

# Drop duplicate key columns if you don’t need them anymore
LOT_data = LOT_data.drop(columns=["partid",  "Dec_ID"])


In [ ]:
# produce df with one value per model per item -------------------------------------------------------
def get_LLM_value_per_item(data):
    grouped = data.groupby(["experiment", "model", "round"])
    prob_sum = grouped["prob_pred"].sum()
    
    # compute weighted mean
    R_score = (grouped["R"].apply(lambda x: (x * data.loc[x.index, "prob_pred"]).sum()) / prob_sum)
    return R_score.reset_index(name="score")


 
model_item_scores_LOT = get_LLM_value_per_item(LOT_data)
model_item_scores_LOT_top_n = get_LLM_value_per_item_top_n(LOT_data, human_col="R")

# Merge them on the grouping keys
model_item_scores_LOT = model_item_scores_LOT.merge(
    model_item_scores_LOT_top_n,
    on=["experiment", "model", "round"],
    how="inner" 
)


In [ ]:
# rename LOT round column to "item" to fit to all_data
model_item_scores_LOT = model_item_scores_LOT.rename(columns = {"round": "item"})

In [ ]:
# Which option is the RISKY one. Convention: True => risky is s2, so the raw
# s1->1 mapping gets flipped (1 - x); False => risky is s1 (no flip).
# Ground truth from Frey table S8 + lotteries.csv choices (R vs Decision_X over
# all participants): risky = Option B (s2) for items 1-15, Option A (s1) for
# items 16-25. NB: the human `Presentation_XZ` order is randomised ~50/50 per
# participant and must NOT be used to set this.
# Regression check: src/preprocessing/check_lot_risky_coding.py
risky_decision_table = {
    1: True,  2: True,  3: True,  4: True,  5: True,  6: True,  7: True,  8: True,  9: True,  10: True,
    11: True, 12: True, 13: True, 14: True, 15: True, 16: False, 17: False, 18: False, 19: False, 20: False,
    21: False, 22: False, 23: False, 24: False, 25: False
}
model_item_scores_LOT["reverse_coded"] = model_item_scores_LOT["item"].map(risky_decision_table)

In [ ]:
# organize outcome dfs

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_LOT], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_LOT], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
# no flipped data

LOT unvollständig!!!

## MPL TASK

In [ ]:
# load data
MPL_data = load_dataframes(task_name="MPL", path = PATH)
MPL_human_data = pd.read_csv("../../data/raw/risk_data/orig_human_data/mpl.csv")

In [ ]:
# normalise answer option sum to one
MPL_data["prob_1"] = np.exp(MPL_data["log_prob_lot_1"])/(np.exp(MPL_data["log_prob_lot_1"]) + np.exp(MPL_data["log_prob_lot_2"]))
MPL_data["prob_2"] = np.exp(MPL_data["log_prob_lot_2"])/(np.exp(MPL_data["log_prob_lot_1"]) + np.exp(MPL_data["log_prob_lot_2"]))

In [ ]:
# filter out probability LLM assigned to real item answer 
MPL_data=filter_pred_prob_including_key_logic(MPL_data, "lot_1_key", "lot_2_key")

In [ ]:
# Merge only selected columns from DFD_human_data
MPL_data = MPL_data.merge(
    MPL_human_data[["partid", "dp", "decision", "R"]],
    left_on=["participant", "problem", "decision"],
    right_on=["partid", "dp", "decision"],
    how="left"
)

# Drop duplicate key columns if you don’t need them anymore
MPL_data = MPL_data.drop(columns=["partid",  "dp"])


In [ ]:
# produce df with one value per model per item -------------------------------------------------------
def get_LLM_value_per_item(data):
    grouped = data.groupby(["experiment", "model", "problem", "decision"])
    prob_sum = grouped["prob_pred"].sum()
    
    # compute weighted mean
    scoreR = (grouped["R"].apply(lambda x: (x * data.loc[x.index, "prob_pred"]).sum()) / prob_sum)
    
    # combine into one DataFrame
    result = pd.concat([scoreR], axis=1).reset_index()
    result.columns = ["experiment", "model", "problem", "decision", "score"]
    
    return result

# produce df with one value per model per item for top n version --------------------------------------------------
def get_LLM_value_per_item_top_n(data, human_col = "R"):
    new_df = (
    data.groupby(["experiment", "model", "problem", "decision"])[[human_col, "prob_pred"]]
      .apply(lambda group: compute_top_n_weighted_score(group, human_col=human_col))
      .reset_index(name="score_top_n")
    )
    return(new_df)

model_item_scores_MPL = get_LLM_value_per_item(MPL_data)
model_item_scores_MPL_top_n = get_LLM_value_per_item_top_n(MPL_data, human_col="R")

# Merge them on the grouping keys
model_item_scores_MPL = model_item_scores_MPL.merge(
    model_item_scores_MPL_top_n,
    on=["experiment", "model", "problem", "decision"],
    how="inner" 
)

In [ ]:
# add item columns to MPL data that fits round and decision number (to have one unique identifier)
model_item_scores_MPL["item"] = model_item_scores_MPL.apply(lambda row: [row["problem"], row["decision"]], axis=1)
model_item_scores_MPL = model_item_scores_MPL.drop(columns=["problem", "decision"])

In [ ]:
# organize outcome dfs
model_item_scores_MPL["reverse_coded"] = False

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, model_item_scores_MPL], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, model_item_scores_MPL], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
# no flipped data

# Save final complete data

In [ ]:
# save data

no_flip_data_rereversed.to_csv('../../data/intermediate/risk_data/LLM_data_proc_prompts_weighed/LLM_weighed_no_flip_data_rereversed.csv', index=False)
no_flip_data_raw.to_csv('../../data/intermediate/risk_data/LLM_data_proc_prompts_weighed/LLM_weighed_no_flip_data_raw.csv', index=False)

flip_data_rereversed.to_csv('../../data/intermediate/risk_data/LLM_data_proc_prompts_weighed/LLM_weighed_flip_data_rereversed.csv', index=False)
flip_data_raw.to_csv('../../data/intermediate/risk_data/LLM_data_proc_prompts_weighed/LLM_weighed_flip_data_raw.csv', index=False)